# Notebook 4 -- Sentinel-2 Vegetation Indices and Remote-Sensing Feature Engineering

**Goal:** prepare Sentinel-2 vegetation-index features for crop-yield prediction.

This notebook gives a progressive implementation path: compute vegetation indices, query Sentinel-2 L2A through STAC, summarize imagery, and merge the resulting features into the crop-yield table.

In [1]:
# Core scientific stack
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Resolve project root whether the notebook is run from /notebooks or from the repository root.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"
MODELS_DIR = PROJECT_ROOT / "models"

for folder in [INTERIM_DIR, PROCESSED_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

RANDOM_STATE = 42


def print_section(title: str):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

# Notebook-aware figure saving -------------------------------------------------
# Every plot generated in this notebook is saved under reports/figures/.
# The counter prevents overwriting plots produced inside loops.
import re

NOTEBOOK_ID = "04_sentinel2_feature_engineering"
_FIGURE_SAVE_COUNTER = 1


def _safe_filename(text: str) -> str:
    text = str(text).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_") or "figure"


def save_figure(fig, name: str = "figure", dpi: int = 180):
    """Save a matplotlib figure to reports/figures with a notebook-specific name."""
    global _FIGURE_SAVE_COUNTER
    filename = f"{NOTEBOOK_ID}_{_FIGURE_SAVE_COUNTER:02d}_{_safe_filename(name)}.png"
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"Saved figure: {path.relative_to(PROJECT_ROOT)}")
    _FIGURE_SAVE_COUNTER += 1
    return path


## 4.1 Loading project helpers and defining paths

In [2]:
from nigeria_crop_yield.data.sentinel2 import vegetation_indices, sentinel_feature_schema, load_sentinel_features

STATE_META_CSV = PROCESSED_DIR / "state_metadata.csv"
SENTINEL_FEATURES_CSV = INTERIM_DIR / "sentinel2_state_crop_season_features.csv"
SENTINEL_TEMPLATE_CSV = INTERIM_DIR / "sentinel2_feature_template.csv"

state_meta = pd.read_csv(STATE_META_CSV)
display(state_meta.head())
print("Sentinel feature file exists:", SENTINEL_FEATURES_CSV.exists())

,state,geopolitical_zone,agroecological_zone,latitude,longitude
0,Abia,South East,Derived Savannah / Humid Forest Transition,5.45,7.52
1,Adamawa,North East,Northern Guinea Savannah,9.33,12.50
2,Akwa Ibom,South South,Humid Forest / Mangrove,5.02,7.92
3,Anambra,South East,Derived Savannah / Humid Forest Transition,6.22,6.94
4,Bauchi,North East,Sudan Savannah,10.31,9.84


Sentinel feature file exists: False


The result `False` only means that the processed Sentinel-2 feature file has not yet been created at: `data/interim/sentinel2_state_crop_season_features.csv`

This is expected at this stage. The notebook has loaded the state metadata containing latitude and longitude values for Nigerian states, but it has not yet queried Sentinel-2 imagery or computed vegetation-index features.
Sentinel-2 imagery is not stored directly in this project because raw satellite images are large. Instead, the workflow is to access Sentinel-2 remotely, extract only the needed bands, compute compact features such as NDVI, EVI, NDWI, SAVI, and cloud-cover summaries, and save those features as a CSV file. So, `False` does not mean Sentinel-2 is unavailable. It only means the feature-extraction step has not yet been run. After the Sentinel-2 extraction cell runs successfully, this file should be created and the check should return `True`.

## 4.2 Understanding vegetation-index formulas

Sentinel-2 L2A common bands:

- `B02` = blue
- `B03` = green
- `B04` = red
- `B08` = near infrared
- `B11`/`B12` = short-wave infrared

NDVI, EVI, NDWI, SAVI, and NDMI are compact indicators of vegetation greenness, canopy vigor, water stress, and moisture state.

In [4]:
# Example with simple reflectance-like values.
red = 0.12
green = 0.16
blue = 0.08
nir = 0.45
swir = 0.22

example_indices = vegetation_indices(red=red, green=green, blue=blue, nir=nir, swir=swir)
display(pd.Series(example_indices, name="example_index_value").to_frame())

,example_index_value
ndvi,0.578947
evi,0.525478
ndwi,-0.475410
savi,0.462617
ndmi,0.343284


## 4.3 Creating a feature schema template

This tell us exactly what the modeling pipeline expects if we later want to run the full remote-sensing extraction.

In [5]:
template = pd.DataFrame(columns=sentinel_feature_schema())
template.to_csv(SENTINEL_TEMPLATE_CSV, index=False)
print("Saved Sentinel-2 feature template:", SENTINEL_TEMPLATE_CSV)
display(template)

Saved Sentinel-2 feature template: c:\Users\Peter\Documents\projects\SHORT PROJECTS\ml-projects\crop-yield-nigeria-geoai\data\interim\sentinel2_feature_template.csv


,state,crop,season,ndvi_mean,ndvi_std,evi_mean,ndwi_mean,savi_mean,cloud_cover_mean,n_observations


## 4.4 Optional STAC query setup

To run this section, install the optional geospatial stack:

```bash
pip install -r requirements-geo.txt
```

The code below is written to fail gracefully if the geospatial dependencies are not installed.

In [ ]:
#pip install -r requirements-geo.txt

  Using cached pytz-2026.2-py2.py3-none-any.whl.metadata (22 kB)
   ---------------------------------------- 0.0/30.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/30.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/30.9 MB ? eta -:--:--
   - -------------------------------------- 0.8/30.9 MB 2.0 MB/s eta 0:00:15
   -- ------------------------------------- 1.6/30.9 MB 3.1 MB/s eta 0:00:10
   -- ------------------------------------- 1.6/30.9 MB 3.1 MB/s eta 0:00:10
   -- ------------------------------------- 1.6/30.9 MB 3.1 MB/s eta 0:00:10
   -- ------------------------------------- 1.6/30.9 MB 3.1 MB/s eta 0:00:10
   -- ------------------------------------- 1.6/30.9 MB 3.1 MB/s eta 0:00:10
   -- ------------------------------------- 1.6/30.9 MB 3.1 MB/s eta 0:00:10
   -- ------------------------------------- 1.6/30.9 MB 3.1 MB/s eta 0:00:10
   -- ------------------------------------- 1.6/30.9 MB 3.1 MB/s eta 0:00:10
   -- --------------------------

  You can safely remove it manually.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
try:
    import planetary_computer
    from pystac_client import Client
    GEO_STACK_AVAILABLE = True
except Exception as exc:
    GEO_STACK_AVAILABLE = False
    print("Optional STAC stack not available yet:", repr(exc))

print("GEO_STACK_AVAILABLE:", GEO_STACK_AVAILABLE)

GEO_STACK_AVAILABLE: True


## 4.5 Querying Sentinel-2 L2A items for a selected state

This cell only queries metadata. It does not download full imagery. I only use it first to confirm that the STAC search works.

In [7]:
SELECTED_STATE = "Kaduna"
DATE_RANGE = "2022-06-01/2022-10-31"
MAX_CLOUD_COVER = 30
BUFFER_DEGREES = 0.35  # approximate bounding box around state centroid for demonstration

if GEO_STACK_AVAILABLE:
    row = state_meta.loc[state_meta["state"].eq(SELECTED_STATE)].iloc[0]
    lon, lat = float(row.longitude), float(row.latitude)
    bbox = [lon - BUFFER_DEGREES, lat - BUFFER_DEGREES, lon + BUFFER_DEGREES, lat + BUFFER_DEGREES]

    catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime=DATE_RANGE,
        query={"eo:cloud_cover": {"lt": MAX_CLOUD_COVER}},
        limit=10,
    )
    items = list(search.items())
    print(f"Items found for {SELECTED_STATE}:", len(items))
    if items:
        item_table = pd.DataFrame([
            {
                "id": item.id,
                "datetime": item.datetime,
                "cloud_cover": item.properties.get("eo:cloud_cover"),
                "platform": item.properties.get("platform"),
            }
            for item in items
        ])
        display(item_table)
else:
    print("Install requirements-geo.txt to run the STAC query.")

Items found for Kaduna: 17


,id,datetime,cloud_cover,platform
0,S2B_MSIL2A_20221024T095029_R079_T32PLS_2024080...,2022-10-24 09:50:29.024000+00:00,16.746928,Sentinel-2B
1,S2B_MSIL2A_20221024T095029_R079_T32PLS_2022102...,2022-10-24 09:50:29.024000+00:00,16.741934,Sentinel-2B
2,S2B_MSIL2A_20221024T095029_R079_T32PKS_2024080...,2022-10-24 09:50:29.024000+00:00,24.255474,Sentinel-2B
3,S2B_MSIL2A_20221024T095029_R079_T32PKS_2022102...,2022-10-24 09:50:29.024000+00:00,24.223019,Sentinel-2B
4,S2A_MSIL2A_20221019T095031_R079_T32PLS_2024082...,2022-10-19 09:50:31.024000+00:00,22.201538,Sentinel-2A
5,S2A_MSIL2A_20221019T095031_R079_T32PLS_2022101...,2022-10-19 09:50:31.024000+00:00,22.180279,Sentinel-2A
6,S2A_MSIL2A_20221019T095031_R079_T32PKT_2024082...,2022-10-19 09:50:31.024000+00:00,22.012329,Sentinel-2A
7,S2A_MSIL2A_20221019T095031_R079_T32PKT_2022101...,2022-10-19 09:50:31.024000+00:00,21.932451,Sentinel-2A
8,S2A_MSIL2A_20221019T095031_R079_T32PKS_2024082...,2022-10-19 09:50:31.024000+00:00,12.198231,Sentinel-2A
9,S2A_MSIL2A_20221019T095031_R079_T32PKS_2022101...,2022-10-19 09:50:31.024000+00:00,12.183002,Sentinel-2A


## 4.6 Remote-sensing feature-extraction scaffold

The full imagery workflow can be computationally heavy. This function shows the intended structure: query items, read required bands, mask clouds, compute vegetation indices, and aggregate them into compact state/season features.

For a small project like this, this kind of clear scaffold is valuable even when raw imagery is too large to bundle.

In [8]:
def summarize_sentinel_features_for_state(
    state: str,
    season: str,
    crop: str | None = None,
    date_range: str = "2022-06-01/2022-10-31",
    max_cloud_cover: float = 30,
) -> dict:
    """Template for Sentinel-2 feature extraction.

    This function intentionally returns a schema-compatible dictionary. To make it fully operational,
    add raster reading with stackstac/rioxarray, apply cloud masks, compute indices from B02/B03/B04/B08/B11,
    then replace the placeholder NaNs with real summary statistics.
    """
    return {
        "state": state,
        "crop": crop if crop is not None else "ALL_CROPS",
        "season": season,
        "ndvi_mean": np.nan,
        "ndvi_std": np.nan,
        "evi_mean": np.nan,
        "ndwi_mean": np.nan,
        "savi_mean": np.nan,
        "cloud_cover_mean": np.nan,
        "n_observations": 0,
    }

example_feature_row = summarize_sentinel_features_for_state("Kaduna", "major", crop="MAIZE")
display(pd.DataFrame([example_feature_row]))

,state,crop,season,ndvi_mean,ndvi_std,evi_mean,ndwi_mean,savi_mean,cloud_cover_mean,n_observations
0,Kaduna,MAIZE,major,NaN,NaN,NaN,NaN,NaN,NaN,0


## 4.7 Loading a real Sentinel-2 feature table if available

After running a full extraction workflow and saving the compact table as `data/interim/sentinel2_state_crop_season_features.csv`, the modeling pipeline will join it by `state`, `crop`, and `season`.

In [9]:
if SENTINEL_FEATURES_CSV.exists():
    sentinel_features = load_sentinel_features(SENTINEL_FEATURES_CSV)
    print(sentinel_features.shape)
    display(sentinel_features.head())
    display(sentinel_features.describe(include="all").T)
else:
    sentinel_features = pd.DataFrame(columns=sentinel_feature_schema())
    print(
        "No real Sentinel-2 feature file found yet. The project can still run with NBS + climate. "
        "To add imagery features, create data/interim/sentinel2_state_crop_season_features.csv using the template."
    )

No real Sentinel-2 feature file found yet. The project can still run with NBS + climate. To add imagery features, create data/interim/sentinel2_state_crop_season_features.csv using the template.


## 4.8 Merging Sentinel-2 features with NBS/climate table

This cell creates an interim merged table if Sentinel-2 features exist. If the feature file is empty or absent, it keeps the base dataset unchanged.

In [10]:
BASE_WITH_CLIMATE = INTERIM_DIR / "nbs_with_climate_features.csv"
NBS_CSV = PROCESSED_DIR / "nbs_crop_yield_state_zone_2022_2023.csv"

if BASE_WITH_CLIMATE.exists():
    base = pd.read_csv(BASE_WITH_CLIMATE)
else:
    nbs = pd.read_csv(NBS_CSV)
    base = (
        nbs[~nbs["is_aggregate"].astype(bool)]
        .query("yield_kg_ha.notna()")
        .merge(state_meta, on="state", how="left")
        .copy()
    )
    base["log_planted_area_ha"] = np.log1p(base["planted_area_ha"].clip(lower=0))

if not sentinel_features.empty and sentinel_features["n_observations"].fillna(0).sum() > 0:
    merged = base.merge(sentinel_features, on=["state", "crop", "season"], how="left")
else:
    merged = base.copy()

OUT = INTERIM_DIR / "nbs_climate_sentinel_features.csv"
merged.to_csv(OUT, index=False)
print("Saved:", OUT)
print(merged.shape)
display(merged.head())

Saved: c:\Users\Peter\Documents\projects\SHORT PROJECTS\ml-projects\crop-yield-nigeria-geoai\data\interim\nbs_climate_sentinel_features.csv
(490, 27)


,season,zone,state,crop,households_reporting_000,planted_area_ha,harvested_area_ha,harvested_quantity_kg,yield_kg_ha,is_aggregate,source_sheet,source_table,geopolitical_zone,agroecological_zone,latitude,longitude,log_planted_area_ha,total_rainfall_mm,mean_tmean_c,mean_tmax_c,mean_tmin_c,mean_rh_percent,mean_wind_speed_m_s,growing_degree_days,heat_stress_days,mean_solar_radiation_kwh_m2_day,total_solar_radiation_kwh_m2
0,major,North Central,Benue,MAIZE,850.39,367795.171270,350398.700331,3.939929e+08,1124.413179,False,Crop Production2_major season,Table 1.1: Main 9 crops highly cultivated in a...,North Central,Southern Guinea Savannah,7.34,8.74,12.815284,2791.10,25.855397,30.575918,21.825959,75.860397,1.950466,11574.44,2.0,18.280082,13344.46
1,major,North Central,FCT,MAIZE,266.34,153246.079216,123737.798835,1.606178e+08,1298.049753,False,Crop Production2_major season,Table 1.1: Main 9 crops highly cultivated in a...,North Central,Southern Guinea Savannah,9.08,7.40,11.939807,2791.66,25.271973,31.020178,20.592260,71.285904,1.552260,11148.54,94.0,19.003315,13872.42
2,major,North Central,Kogi,MAIZE,640.86,327793.065371,309235.626289,4.257146e+08,1376.667527,False,Crop Production2_major season,Table 1.1: Main 9 crops highly cultivated in a...,North Central,Southern Guinea Savannah,7.80,6.74,12.700141,2922.19,25.946644,30.472301,21.926014,77.683192,1.599945,11641.05,2.0,17.665493,12895.81
3,major,North Central,Kwara,MAIZE,398.11,307483.872742,301489.864767,5.002011e+08,1659.097522,False,Crop Production2_major season,Table 1.1: Main 9 crops highly cultivated in a...,North Central,Southern Guinea Savannah,8.50,4.55,12.636181,3120.12,25.177890,30.194986,21.067233,78.085986,1.852548,11079.86,11.0,18.085219,13202.21
4,major,North Central,Nasarawa,MAIZE,520.48,416547.122485,360177.787028,4.213675e+08,1169.887511,False,Crop Production2_major season,Table 1.1: Main 9 crops highly cultivated in a...,North Central,Southern Guinea Savannah,8.54,8.32,12.939757,2904.55,26.104479,31.606562,21.643397,71.356986,1.675493,11756.27,113.0,18.834699,13749.33


## 4.9 Remote-sensing feature diagnostics

When real Sentinel-2 features are available, this section checks feature distributions and missingness.

In [11]:
remote_cols = ["ndvi_mean", "ndvi_std", "evi_mean", "ndwi_mean", "savi_mean", "cloud_cover_mean", "n_observations"]
available_remote_cols = [c for c in remote_cols if c in merged.columns]

if available_remote_cols:
    miss = merged[available_remote_cols].isna().mean().mul(100).round(2).to_frame("missing_percent")
    display(miss)

    numeric_remote_cols = [c for c in available_remote_cols if pd.api.types.is_numeric_dtype(merged[c])]
    for col in numeric_remote_cols:
        if merged[col].notna().sum() == 0:
            continue
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(merged[col].dropna(), bins=30)
        ax.set_title(f"Distribution of {col}")
        ax.set_xlabel(col)
        ax.set_ylabel("Count")
        save_figure(fig, "figure_01")
        plt.show()
else:
    print("No remote-sensing columns available yet.")

No remote-sensing columns available yet.
